# Short Guide to the MPT Python Package by Felix Guischard

## Dependencies

- numpy
- matplotlib
- anytree
- pandas
- seaborn
- msmhelper
- tqdm
- sklearn
- graph
- scipy
- numba
- mdtraj
- prettypyplot

## Preparation

Load the MPT package, make sure the path is in your $PYTHONPATH or you employ:

```import sys
sys.path.insert(0,'/path/to/MPT')```

In [1]:
import MPT

Load other required packages

In [2]:
import numpy as np

Load required data

In [11]:
# Directory where the files are stored
source_directory = "/data/evaluation/MPP/stochastic_MPP_Felix/data_source/"

# Microstate trajectory - Must be 1-based (microstate indices starting with 1)
traj_path = source_directory + "hp35.selected_contacts.gaussian10f_microstates_pcs5_p153"
traj = np.loadtxt(traj_path, dtype=int)

# Load fraction of native contacts (fnc) trajectory (values between 0 and 1)
feature_traj_path = source_directory + "hp35.mindists2.gaussian10f.q"
feature_traj = np.loadtxt(feature_traj_path)

# native contacts trajectory (each contact resolved)
multi_feature_traj_path = source_directory + "hp35.mindist2.gaussian10f"
multi_feature_traj = np.loadtxt(multi_feature_traj_path)
# convert the trajectory to boolean, indicating if a contact is formed (True) or not (False)
multi_feature_traj_bool = multi_feature_traj < 0.45

# In cluster_file, the contact clusters are defined
cluster_file = source_directory + "hp35.mindist2.mosaic_clusters"

Define the lagtime

In [8]:
lagtime = 50 # = 10 ns

## Initialization of the Kernel

The kernel performs the decision step in lumping. The lumping approach is determined here and it can take the following parameters:

- method:
  - "n": take the **param** most probable option into account (**param**: integer >= 1, default: 1)
  - "p": take the most probable options into account, which sum up to **param**. (**param**: float 0 < param <= 1, default: 1)
- param: see method
- cutoff: Only consider transition probabilities, which are at least **cutoff** * *T_ij,max* (float, 0 <= cutoff <= 1, default: 0)
- similarity: dynamic similarity metric (*v_ij*) used to determine lumping partner
  - "P": transition probabilities
  - "KL": Kullback-Leibler divergence of transition probability vectors + softmax function
  - "JS": Jensen-Shannon divergence of transition probability vectors + softmax function
- a, b, c: Weighting factors for transition probabilities, KL/JS probabilities and feature, respectively. The respective part is omitted when a, b or c = 0. (float, default: a=1, b=0, c=0)
- term: (default: "*")
  - "+": sum up the values of ransition probabilities, KL/JS probabilities and feature
  - "*": multiply the values of ransition probabilities, KL/JS probabilities and feature

If the kernel is initialized without any parameter, the reference lumping is performed.

In [10]:
mpt_kernel = MPT.kernel.MPTKernel()
smpt_KL_kernel = MPT.kernel.MPTKernel(param=2, similarity="KL", a=0, b=1)

## Initialization of the Feature Kernel

The feature kernels work out the feature for each lumping step. There are two kernel objects: The FeatureKernel for a one-dimensional feature like the fraction of natve contacts (fnc) and the MultiFeatureKernel for multidimensional features like the vectors of fractions of native contacts. The both require the feature trajectory and the microstate trajectory as positional arguments.

FeatureKernel:

- sigma: sigma of the Gaussian function (default: 0.13)
- b: exponent of the Gaussian function (default: 2)

MultiFeatureKernel:

- similirity:
  - "KL": similarity based on the Kullback-Leibler divergence and the softmax function
  - "JS": similarity based on the Jensen-Shannon divergence and the softmax function

In [14]:
feature_kernel = MPT.kernel.FeatureKernel(feature_traj, traj, sigma=0.05)
multi_feature_kernel = MPT.kernel.MultiFeatureKernel(multi_feature_traj_bool, traj, similarity="JS")

## Initialization of the MPT object

The MPT object holds all the data and is used to perform the lumping.

### Positional arguments:

- traj: microstate trajectory
- tlag: lagtime
- feature_traj: feature trajectory (e.g. fnc)

### Keyword arguments:

- macrostate_thresholds: tuple of minimum population and minimum metastability *T_ii*. (default: (0.005, 0.5))
- quiet: Whether to print progressbars or not. (default: False)

### Attributes:

As this class holds all the data, it's got many attributes to make the data available. To see the most important ones, please review the \_\_init\_\_ function and the assign_macrostates method.

### Methods:

Again, as this is the central class, it contains all the methods required for performing a lumping and many for its analysis. For stochastic lumpings, make sure that n_i is set in order to analyse the lumping you want. For convenience, the class provides save and load functions in order to save and load lumpings.

In [19]:
mpt = MPT.MPT(traj, lagtime, feature_traj)
smpt_kl_js = MPT.MPT(traj, lagtime, feature_traj)

## Performing a Lumping - the mpt.mpt() Method

The mpt.mpt() method performs the lumping.

### Positional arguments:

- kernel: MPTKernel object

### Keyword arguments:

- feature_kernel: FeatureKernel or MultiFeatureKernel object. If 1, no feature is applied. (default: 1)
- n: number of lumpings to perform. Used to determine the number of stochastic lumpings to perform. (default: 1)

In [20]:
mpt.mpt(mpt_kernel)
smpt_kl_js.mpt(smpt_KL_kernel, feature_kernel=multi_feature_kernel, n=30)

Clustering ...


100%|██████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.64it/s]


Assigning macrostates ...


100%|██████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.66it/s]


Clustering ...


100%|████████████████████████████████████████████████████████████████████| 30/30 [01:12<00:00,  2.43s/it]


Assigning macrostates ...


100%|████████████████████████████████████████████████████████████████████| 30/30 [00:04<00:00,  6.07it/s]


## Important Attributes

- mpt.ref: Direct access to the reference lumping
- mpt.Z: Z matrix of shape (*N*-1, 4), *N* is the number of microstates, which contains the lumping tree in compact form. Row *i* defines the intermediate state with index *N* + *i* from the lumping of the two states in the first two fields of that row. The other two fields correspond to the metastability and the population of that state. This matrix can be saved and loaded (MPT.save_Z() and MPT.from_Z()). Further reference: https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html
- mpt.tree: Tree structure, which holds all states and all information associated with it. Used for macrostate assignemnent.
- mpt.n_i: Index of the lumping to analyse when using plot methods

## Important Methods

- mpt.plot(out_file): Plot a dendrogram
- mpt.plot_implied_timescales(out_file): Plot implied timescales plot
- mpt.plot_sankey(out_file): Plot Sankey diagram
- mpt.plot_contact_rep(multi_feature_traj, cluster_file, out_file): Plot contacts representation plot

## Further Useful Functions

- MPT.plot.report: produces a latex file with a number of analyses and numbers
- MPT.plot.report_stochastic: Same for stochastic lumping